In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler




df = pd.read_excel('./deskryptory_z_cas_2.xlsx')
df_klasy = pd.read_excel('./Klasy_FR.xlsx')
df_klasy.set_index("cas", inplace=True)

df.set_index(df.columns[0], inplace=True)

X = df.select_dtypes(include=[np.number]).dropna()


pipeline = Pipeline([
    ('drop_low_variance', VarianceThreshold(threshold=0.01)), 
    ('scaler', StandardScaler()),                             # PCA bardzo lubi wyskalowane dane!
    ('pca', PCA())
])


X_pca = pipeline.fit_transform(X)
fitted_pca = pipeline.named_steps["pca"]

explained_variance = fitted_pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)




plt.figure(figsize=(10, 5))
plt.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.5, align='center', label='Wariancja pojedyncza')
plt.step(range(1, len(cumulative_variance) + 1), cumulative_variance, where='mid', label='Wariancja skumulowana', color='red')
plt.ylabel('Procent wyjaśnionej wariancji')
plt.xlabel('Główne składowe (PC)')
plt.title('Wykres osypiska (Scree Plot) dla deskryptorów')
plt.legend(loc='best')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

pca_df = pd.DataFrame(data=X_pca[:, :4], columns=['PC1', 'PC2','PC3','PC4'], index=X.index)

pca_df["klasa_chemiczna"] = pca_df.index.map(df_klasy["klasa_chemiczna"])

x_var = "PC1"
y_var = "PC2"

plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=pca_df,
    x=x_var,
    y=y_var,
    hue="klasa_chemiczna",
    palette="tab10",
    alpha=0.8,
    edgecolor="w",
    s=70,
)

plt.title("Wykres rozrzutu PCA (Score Plot)")
plt.xlabel(f"{x_var}")
plt.ylabel(f"{y_var}")
plt.grid(True, linestyle="--", alpha=0.5)

plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Klasa chemiczna")

for i, txt in enumerate(pca_df.index):
    plt.annotate(
        txt,
        (pca_df[x_var].iloc[i], pca_df[y_var].iloc[i]),
        fontsize=8,
        alpha=0.7,
        xytext=(3, 3),
        textcoords="offset points",
    )

plt.tight_layout()
plt.show()
